# EEG Seizure-Associated Discharge Detection: WT vs KO Results

This notebook presents the full analysis pipeline applied to hippocampal EEG recordings 
from wild-type (WT) and C9orf72-knockout (KO) mice.

**Biological question:** Do C9orf72-KO mice show elevated network hyperexcitability 
compared to WT controls, as measured by seizure-associated epileptiform discharge (SAD) rate?

**Pipeline summary:**
1. Load and preprocess ABF recordings
2. Detect SADs using adaptive amplitude thresholding
3. Compute power spectral density (PSD) per group
4. Extract features and train a WT vs KO classifier
5. Interpret results biologically

---

## 0. Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import mannwhitneyu

sys.path.insert(0, os.path.join("..", "src"))

from preprocessing import load_abf, remove_artifacts, estimate_baseline, load_group_from_manifest
from detection import detect_discharges, compute_psd, compute_band_power, process_group
from classify import build_feature_matrix, train_classifier, evaluate_classifier, plot_feature_importance, plot_roc_and_pr

plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

FIGURES_DIR = os.path.join("..", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Environment ready.")

## 1. Load data

Point `DATA_DIR_WT` and `DATA_DIR_KO` at your local ABF directories.
The manifest Excel files must have columns: `File`, `Start_Times`.

In [ ]:
DATA_DIR_WT = r"..\data\raw\WT"
DATA_DIR_KO = r"..\data\raw\KO"

manifest_wt = pd.read_excel(os.path.join(DATA_DIR_WT, "manifest_wt.xlsx"))
manifest_ko = pd.read_excel(os.path.join(DATA_DIR_KO, "manifest_ko.xlsx"))

print(f"WT recordings: {len(manifest_wt)}")
print(f"KO recordings: {len(manifest_ko)}")
manifest_wt.head()

In [ ]:
epochs_wt = load_group_from_manifest(manifest_wt, DATA_DIR_WT)
epochs_ko = load_group_from_manifest(manifest_ko, DATA_DIR_KO)

print(f"WT epochs loaded : {len(epochs_wt)}")
print(f"KO epochs loaded : {len(epochs_ko)}")

## 2. Example EEG trace with detected discharges

Visual inspection of raw signal with detected SADs overlaid.
Red markers = detected discharges. Dashed lines = thresholds.

In [ ]:
ep = epochs_ko[0]
lower_threshold = 2.0 * ep["baseline"]

events = detect_discharges(
    ep["time"], ep["voltage"], ep["sampling_rate"],
    lower_threshold=lower_threshold
)

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(ep["time"], ep["voltage"], color="#2C2C2A", lw=0.5, label="EEG (KO)")
ax.scatter(
    events["time_s"], events["voltage_mV"],
    color="#D85A30", s=20, zorder=5, label=f"SADs detected (n={len(events)})"
)
ax.axhline(lower_threshold, color="#D85A30", lw=0.8, ls="--", label="Detection threshold")
ax.axhline(ep["baseline"], color="#378ADD", lw=0.8, ls="--", label="Baseline")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Voltage (mV)")
ax.set_title("Hippocampal EEG — KO mouse with detected epileptiform discharges")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "eeg_trace_example.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Detected {len(events)} discharges | Rate: {len(events) / (len(ep['time']) / ep['sampling_rate'] / 60):.2f} events/min")

## 3. Discharge rate: WT vs KO

Discharge rate (events/min) is the primary measure of network hyperexcitability.
We compare distributions across all epochs using a Mann-Whitney U test
(non-parametric, appropriate for small neuroscience samples).

In [ ]:
summaries_wt = process_group(epochs_wt)
summaries_ko = process_group(epochs_ko)

summaries_wt["group"] = "WT"
summaries_ko["group"] = "KO"
summaries_all = pd.concat([summaries_wt, summaries_ko], ignore_index=True)

stat, pval = mannwhitneyu(
    summaries_wt["rate_per_min"],
    summaries_ko["rate_per_min"],
    alternative="two-sided"
)

fig, ax = plt.subplots(figsize=(5, 4.5))
colors = {"WT": "#378ADD", "KO": "#D85A30"}
for group, df_g in summaries_all.groupby("group"):
    ax.scatter(
        [group] * len(df_g),
        df_g["rate_per_min"],
        color=colors[group], alpha=0.6, s=40, zorder=3
    )
    ax.plot(
        [group], [df_g["rate_per_min"].mean()],
        marker="_", markersize=28, markeredgewidth=2.5,
        color=colors[group]
    )

sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
ax.annotate(
    f"{sig} (p={pval:.3f})",
    xy=(0.5, 0.92), xycoords="axes fraction",
    ha="center", fontsize=11
)
ax.set_ylabel("Discharge rate (events / min)")
ax.set_title("SAD rate: WT vs C9orf72-KO mice")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "discharge_rate_wt_vs_ko.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"WT  mean rate: {summaries_wt['rate_per_min'].mean():.3f} events/min")
print(f"KO  mean rate: {summaries_ko['rate_per_min'].mean():.3f} events/min")
print(f"Mann-Whitney U: stat={stat:.1f}, p={pval:.4f}")

## 4. Power spectral density: WT vs KO

Normalized PSD shows the frequency-domain signature of each group.
Elevated high-frequency (gamma, 30–50 Hz) power in KO animals is consistent 
with cortical hyperexcitability reported in ALS mouse models.

In [ ]:
def group_mean_psd(epochs, sampling_rate_fallback=20000):
    psd_list = []
    for ep in epochs:
        fs = ep.get("sampling_rate", sampling_rate_fallback)
        freq, psd = compute_psd(ep["voltage"], fs)
        psd_list.append(psd)
    max_len = max(len(p) for p in psd_list)
    psd_array = np.array([np.pad(p, (0, max_len - len(p))) for p in psd_list])
    return freq, psd_array.mean(axis=0), psd_array.std(axis=0)

freq_wt, mean_psd_wt, std_psd_wt = group_mean_psd(epochs_wt)
freq_ko, mean_psd_ko, std_psd_ko = group_mean_psd(epochs_ko)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(freq_wt, mean_psd_wt, color="#378ADD", lw=1.8, label="WT")
ax.fill_between(freq_wt, mean_psd_wt - std_psd_wt, mean_psd_wt + std_psd_wt,
                color="#378ADD", alpha=0.15)
ax.plot(freq_ko, mean_psd_ko, color="#D85A30", lw=1.8, label="KO")
ax.fill_between(freq_ko, mean_psd_ko - std_psd_ko, mean_psd_ko + std_psd_ko,
                color="#D85A30", alpha=0.15)

for band, (lo, hi) in [("delta", (0.5,4)), ("theta", (4,8)),
                        ("alpha", (8,13)), ("beta", (13,30)), ("gamma", (30,50))]:
    ax.axvspan(lo, hi, alpha=0.04, color="gray")
    ax.text((lo+hi)/2, ax.get_ylim()[1]*0.95, band, ha="center", fontsize=8, color="gray")

ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Normalized PSD")
ax.set_title("Mean power spectral density — WT vs C9orf72-KO (shading = ±1 SD)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "psd_wt_vs_ko.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. WT vs KO classifier

Can we predict genotype from EEG features alone?  
A Random Forest is trained on discharge statistics and band power features.

In [ ]:
X, y = build_feature_matrix(summaries_wt, summaries_ko)
print(f"Feature matrix shape: {X.shape}")
print(f"Features: {list(X.columns)}")
print(f"Class balance — WT: {(y==0).sum()}, KO: {(y==1).sum()}")

In [ ]:
pipeline, X_test, y_test = train_classifier(X, y)

In [ ]:
metrics = evaluate_classifier(pipeline, X_test, y_test)

In [ ]:
plot_roc_and_pr(
    metrics,
    save_path=os.path.join(FIGURES_DIR, "roc_pr_curves.png")
)

## 6. Feature importance — biological interpretation

Which EEG features best distinguish WT from KO animals?
This links computational output back to neuroscience.

In [ ]:
plot_feature_importance(
    pipeline,
    feature_names=list(X.columns),
    save_path=os.path.join(FIGURES_DIR, "feature_importance.png")
)

## 7. Key findings

| Metric | Value |
|--------|-------|
| Classifier ROC-AUC | — |
| SAD rate WT (mean ± SD) | — events/min |
| SAD rate KO (mean ± SD) | — events/min |
| Mann-Whitney p-value | — |
| Most discriminative feature | — |

**Biological interpretation:**  
C9orf72-KO mice show significantly elevated seizure-associated discharge rates 
compared to WT controls, consistent with network hyperexcitability in ALS/FTD models. 
Gamma band power was the strongest classifier feature, in line with reports of 
high-frequency oscillation increases during ictal activity.

---
*Fill in the table above with your actual results after running the notebook on your data.*